# 事前準備：共通コードの実行
* このノートブックに接続したら，まずは以下の2つの共通コード（コードAとコードB）を実行する
* これらの共通コードを実行しないと，それ以降のコードが実行できないので注意する
* また，コードA，コードB，及びコードC は，ノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 共通コードの詳細についての説明は割愛する（簡単な説明は[第2回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)を参照）

In [ ]:
# コードA：日本語化ライブラリ導入
! pip install japanize-matplotlib | tail -n 1

In [ ]:
# コードB：共通事前処理

# B1:余分なワーニングを非表示にする
import warnings
warnings.filterwarnings('ignore')

# 必要ライブラリのimport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # matplotlib日本語化対応
import seaborn as sns

# B2:データフレーム表示用関数
from IPython.display import display

# B3:表示オプション調整
np.set_printoptions(suppress = True, precision = 3) #numpyの浮動小数点の表示精度
pd.options.display.float_format = '{:.3f}'.format #pandasでの浮動小数点の表示精度
pd.set_option('display.max_columns', None) #データフレームですべての列データを表示

# B4:グラフのデフォルトフォント指定
plt.rcParams['font.size'] = 14

# 乱数の種
random_seed = 0

In [ ]:
# コードC：データのダウンロード
!curl -L -o wholesale.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/09/wholesale.csv
!curl -L -o iris.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/09/iris.csv

# 目的と到達目標

## 目的
* クラスタリングの目的と手法を理解する

## 到達目標
* クラスタリングの目的を説明できる
* k-means法によるクラスタリングを実践できる



# 教師なし学習

## 教師あり学習と教師なし学習
* 機械学習の種類
  * 教師あり学習，教師なし学習，強化学習，半教師あり学習
* モデルの学習ために使うデータ： 訓練データ（あるいはトレーニングデータ）と呼ぶ
* 教師あり学習と教師なし学習の違い
  * 教師あり学習の訓練データには正解データが含まれる
  * 教師なし学習の訓練データには正解データが含まれない


## 教師なし学習におけるモデルの学習
* 目的変数と説明変数の関係性を表現しようとするモデルを学習するのではなく，データそのものに着目しそこに隠れた構造や法則を見つけ出すためのモデルを学習する
* データ自体の特徴や法則を見つける
* すなわち，データ $𝑥_1, 𝑥_2, \ldots, 𝑥_𝑛$ からモデルを作る（学習する）

  <img src="./fig/unsupervised_learning.jpg" width="600">

## 教師なし学習とホールドアウト法
* ホールドアウト法は教師あり学習で用いられる手法
* 教師なし学習では正解データ（教師データ）がないので，モデルの予測値と比べるといったことはできない
* したがって，教師なし学習ではデータを分割せずに，モデルを学習する
* ただし，モデルを学習する際には何らかの妥当性を示す指標を用いる


## 教師なし学習の代表的な問題設定
* クラスタリング
  * 目的：多数のデータをいくつかの類似グループに分類すること
  * 顧客のセグメンテーションやターゲティングを行うときなどに使われる
* 次元削減
  * 目的：元のデータがもつ情報をなるべく減らさずに，変数（特徴量）の数を減らすこと
  * データの変数が多い場合に使われる
* マーケットバスケット分析（アソシエーションルール）
  * 目的：商品を買うときの組み合わせで，どの組合せが多いかなどを明らかにすること

# クラスタリング

## クラスタリングのイメージ
* 特徴量が2つのデータ（2次元データ）は散布図で描ける
* 特徴量が2つのデータの例： 生徒の数学と英語の試験の点数，お店の来店頻度と購入金額など
* 散布図上での点が近いもの同士を同じグループにする
* このグループのことを「クラスタ」と呼ぶ

  <img src="./fig/illustration_of_clustering.jpg" width="450">

## なぜクラスタリングが必要か？
* クラスタリングは，マーケティングなどで，既存の顧客をグループ分けする際に使われることがある
* 例：自社サービスに関するキャンペーンを行い，既存顧客に告知のためのダイレクトメールを送りたい
  * 全顧客に全く同じ内容のメールを送っても顧客によっては全く響かない（無駄な取り組み）
  * そこで，顧客情報を特徴量として，いくつかのグループに分ける（クラスタリング）
  * この結果から，各グループの共通点を調べ，そのグループの顧客の琴線に触れるようなダイレクトメールを送ることで，キャンペーン告知などの効果を最大限に高めることができる

# 代表的なクラスタリングの手法 ～k-means法～
k-means法（k-平均法）は，いくつのクラスタに分けたいかを指定して，データ全体を指定したクラスタ数に分割する．

## k-means法の手順
1. クラスタ数kを決める⇒k個のクラスタ番号0～k-1を定める
2. k個のデータ（乱数）をランダムに生成し，各クラスタの重心点とする
3. 各データに対し，最も近い重心点に対応するクラスタ番号を割り当てる
4. 各クラスタに属するデータの平均値を計算し，新しい重心点とする
5. 手順3と4を繰り返す： 重心点が動かなく（移動距離が十分に小さく）なるか，繰返し上限回数に達したら繰り返し終了


## 例を用いた k-means法のイメージ
* クラスタ数kをk=3とする
* クラスタ番号は 0, 1, 2 
* 3個の重心点（下図の×印）をランダムに生成

  <img src="./fig/k-means1.jpg" width="300">

* 各データに対し，最も近い重心点に対応するクラスタ番号を割り当てる

  <img src="./fig/k-means2.jpg" width="300">

* 各データに対し，最も近い重心点に対応するクラスタ番号を割り当てる

  <img src="./fig/k-means3.jpg" width="370">

* 各クラスタに属するデータの平均値を計算する
* この平均値を新しい重心点とする

  <img src="./fig/k-means4.jpg" width="300">

* 各データに対し，最も近い重心点に対応するクラスタ番号を割り当てる

  <img src="./fig/k-means5.jpg" width="350">

* 各クラスタに属するデータの平均値を計算する
* この平均値を新しい重心点とする

  <img src="./fig/k-means6.jpg" width="300">

* 各データに対し，最も近い重心点に対応するクラスタ番号を割り当てる

  <img src="./fig/k-means7.jpg" width="370">

* 各クラスタに属するデータの平均値を計算する
* この平均値を新しい重心点とする

  <img src="./fig/k-means8.jpg" width="300">

* 各データに対し，最も近い重心点に対応するクラスタ番号を割り当てる
* 重心点が動かなくなったので終了

  <img src="./fig/k-means9.jpg" width="370">


## k-means法によるモデルの学習で用いられる損失関数
* これまで扱ってきたモデルの学習で用いられる損失関数
  * 線形回帰モデルの学習： 平均二乗誤差
  * 分類木モデルの学習： 不純度
  * ロジスティック回帰モデルの学習： 平均交差エントロピー誤差関数
* k-means法によるモデルの学習では「歪み尺度」を損失関数として用いる
  * 具体的な歪み尺度には様々なものがある
  * 代表的な歪み尺度に，クラスタ内誤差平方（SSE: Sum of Squared Errors）がある

## クラスタ内誤差平方（SSE: Sum of Squared Errors）

### SSEの定義
- 各クラスタに属するデータと重心までの距離の2乗の和として定義： $\displaystyle{\sum_{i=0}^{k-1}\sum_{j=1}^{n} \Delta_{i,j} \left\| \boldsymbol{x}_j-\boldsymbol{\mu}_i \right\|^2}$
  - $\Delta_{i,j}$：データ $\boldsymbol{x}_j$ がクラスタ $i$ に属していたら1，それ以外は0
  - $\boldsymbol{\mu}_i$：クラスタ $i$ の重心
  - $\left\|\boldsymbol{x}_j-\boldsymbol{\mu}_i\right\|^2$：ベクトル $(\boldsymbol{x}_j-\boldsymbol{\mu}_i)$ の大きさの2乗（後述）
- クラスタ数とデータ数が同じときSSEは最小値0をとる

### ベクトルの大きさの2乗
- ベクトル $\boldsymbol{a}=(a_1,a_2,a_3)$ の大きさ： $\|\boldsymbol{a}\|=\sqrt{a_1^2+a_2^2+a_3^2}$

- ベクトル $\boldsymbol{a}=(a_1,a_2,a_3)$ の大きさの2乗： $\|\boldsymbol{a}\|^2=a_1^2+a_2^2+a_3^2$

- ベクトル $\boldsymbol{a}=(a_1,a_2,\cdots,a_n)$ の大きさの2乗： $\|\boldsymbol{a}\|^2 = a_1^2+a_2^2+\cdots+a_n^2$

- 例：ベクトル $(2,-1,3)$ の大きさの2乗： $2^2+(-1)^2+3^2 = 4+1+9 = 14$

### SSEの計算例

- 3つのクラスタのいずれかに属する8個の2次元データ $(x,y)$ が下表のように与えられている
- このときのSSEを答えよ

| クラスタ0 | クラスタ1 | クラスタ2 |
|---|---|---|
| $(2,-2)$ | $(3,1)$ | $(3,2)$ |
| $(0,-1)$ | $(2,0)$ | $(1,2)$ |
| $(1,0)$ | $(1,2)$ | - |

- 各クラスタの重心（平均）を計算
  - $\displaystyle{\boldsymbol{\mu}_0 = \left(\frac{2+0+1}{3}, \frac{-2-1+0}{3} \right) = (1,-1)}$
  - $\displaystyle{\boldsymbol{\mu}_1 = \left(\frac{3+2+1}{3}, \frac{1+0+2}{3} \right) = (2,1)}$
  - $\displaystyle{\boldsymbol{\mu}_2 = \left(\frac{3+1}{2}, \frac{2+2}{2} \right) = (2,2)}$
- クラスタ0に属するデータと重心$\boldsymbol{\mu}_0$までの距離の2乗を計算
  - $\displaystyle{\|(2,-2)-(1,-1)\|^2 = \|(1,-1)\|^2 = 1^2+(-1)^2 = 2}$
  - $\displaystyle{\|(0,-1)-(1,-1)\|^2 = \|(-1,0)\|^2 = (-1)^2+0^2 = 1}$
  - $\displaystyle{\|(1,0)-(1,-1)\|^2 = \|(0,1)\|^2 = 0^2+1^2 = 1}$

- クラスタ1に属するデータと重心$\boldsymbol{\mu}_1$までの距離の2乗を計算
  - $\displaystyle{\|(3,1)-(2,1)\|^2 = \|(1,0)\|^2 = 1^2+0^2 = 1}$
  - $\displaystyle{\|(2,0)-(2,1)\|^2 = \|(0,-1)\|^2 = 0^2+(-1)^2 = 1}$
  - $\displaystyle{\|(1,2)-(2,1)\|^2 = \|(-1,1)\|^2 = (-1)^2+1^2 = 2}$

- クラスタ2に属するデータと重心$\boldsymbol{\mu}_2$までの距離の2乗を計算
  - $\displaystyle{\|(3,2)-(2,2)\|^2 = \|(1,0)\|^2 = 1^2+0^2 = 1}$
  - $\displaystyle{\|(1,2)-(2,2)\|^2 = \|(-1,0)\|^2 = (-1)^2+0^2 = 1}$

| クラスタ0 | 距離の2乗 | クラスタ1 | 距離の2乗 | クラスタ2 | 距離の2乗 |
|---|---:|---|---:|---|---:|
| $(2,-2)$ | 2 | $(3,1)$ | 1 | $(3,2)$ | 1 |
| $(0,-1)$ | 1 | $(2,0)$ | 1 | $(1,2)$ | 1 |
| $(1,0)$ | 1 | $(1,2)$ | 2 | - | - |

- 距離の2乗の和がSSEとなる： $2+1+1+1+1+2+1+1=10$


## k-maens法の限界
* k-maens法はデータをクラスタに分けるだけ
* 各クラスタに属するデータがどのような特徴を持っているかなどはわからない
* しかし，各クラスタの特徴を把握することは非常に重要
* そこで，一般にはクラスタリングの結果を別の手法等を用いて分析することで各クラスタの特徴を明らかにしていく

# クラスタリングによる分析例：顧客データ

## 用いるデータと分析の目的
* 例として，csvファイル「wholesale.csv」を用いる
  * 出典: 須藤秋良, 株式会社フレアリンク: スッキリわかるPythonによる機械学習入門, インプレス, 2020
* ここで利用するデータは出典（教科書）のデータを加工している
* オリジナルのデータは，「Wholedf customers Data Set (卸売業者の顧客データ)」と呼ばれているデータセット
  * ポルトガルの卸売業者の顧客データで，商品カテゴリ毎に顧客単位の年間支出額が入っている
  * オリジナルのデータのURL: https://archive.ics.uci.edu/dataset/292/wholesale+customers
* 以降，このデータのことを「顧客データ」と呼ぶ
* 今回は，クラスタリングの代表的手法である「k-means法」を使って，顧客データをクラスタリングすることを考える

## データの読み込み
* 顧客データを読み込んで，内容を確認する
* 各列名と意味は以下のとおり
  
|**列名**| **意味** |
|:--|:--|
|生鮮食品| 生鮮食品の年間支出額 |
|乳製品| 乳製品の年間支出額 |
|食料雑貨品| 食料雑貨品の年間支出額 |
|冷凍食品| 冷凍食品の年間支出額 |
|洗剤・紙製品| 洗剤と紙製品の年間支出額 |
|惣菜| 惣菜の年間支出額 |
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，ファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `display`関数を使ってデータ（`df`の内容）を表示

In [ ]:
# データの読み込み
df = pd.read_csv('wholesale.csv')
display(df)

## 実習：各特徴量の統計量
* 以下のコードで，読み込んだデータの各特徴量（生鮮食品～惣菜）の基本的な統計量を確認する
* 実行結果から，特徴量によって平均値や標準偏差（ばらつき）が大きく異なっていることがわかる
* 実行結果の内容（読み方）については，[第2回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)を参照

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `describe`メソッドで`df`の基本的な統計量を計算し，その結果を`display`関数を使って表示

**［実行結果］**

  <img src="./fig/exercise_describe_wholesale.jpg" width="500">

In [ ]:
# データの読み込み



## データの前処理
* 教師なし学習では，正解データを使ってモデルの性能を評価することができない
* そのため，訓練データとテストデータに分割する必要はないことに注意しておく

### 実習：欠損値の確認
* 以下のコードで，データに欠損値がないか確認する
* 実行結果から，顧客データには欠損値がないことがわかる

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `isnull`メソッドと`sum`メソッドを組み合わせて，各列の欠損値の数を計算して，`display`関数で表示する
  * [第4回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook04.ipynb)で用いたコードと同様

**［実行結果］**

  <img src="./fig/exercise_missing_wholesale.jpg" width="180">

In [ ]:
# 欠損値の確認



### 実習：データの標準化
* これまで扱ってきたロジスティック回帰モデル（分類）や線形回帰モデル（回帰）と同様に，クラスタリング（k-means法）においても，データの標準化を行う
* ただし，クラスタリングは教師なし学習なので，目的変数と説明変数，あるいは訓練データとテストデータといったデータの分割は行わない
* したがって，データ全体（`df`）を使ってデータの標準化を行う

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `preprocessing`モジュールの`StandardScaler`クラスの読み込み
* 4行目: `StandardScaler`クラスのオブジェクトを生成し，変数`sc`に代入
* 5行目: `sc`に対し，`fit`メソッドを適用する（引数は`df`）
* 6行目: `sc`に対する`transform`メソッドで`df`を標準化し，その結果を変数`df_s`に代入
* 7行目: `DataFrame`関数で，`df_s`をDataFrameに変換し，その結果を変数`df_cluster`に代入
  * 引数`columns=df.columns`で，変換したDataFrameの列名を「生鮮食品」～「惣菜」に指定
* 8行目: `display`関数を使ってデータ（`df_cluster`の内容）を表示

**［実行結果］**

  <img src="./fig/exercise_splitting_wholesale.jpg" width="500">

In [ ]:
# データの標準化





df_cluster = pd.DataFrame(df_s, columns=df.columns)
display(df_cluster)

## モデルの学習（k-means法によるクラスタリング）
* k-means法によるクラスタリングは，`sklearn`（scikit-learn）の`cluster`モジュールにおける`KMeans`クラスを使う
* k-means法によるクラスタリングの流れ:
  * `KMeans`クラスをインポートする
  * インポートの書式: `from sklearn.cluster import KMeans`
  * `KMeans`クラスのオブジェクトを生成し，変数に代入する
  * オブジェクト生成の書式: `変数 = KMeans(n_clusters=[クラスタ数], random_state=[乱数の種])`
  * 乱数は初期値（重心）を決める際に使用する（乱数によって結果が変わることに注意する）
  * モデルを学習する（クラスタリングを行う）
  * モデルの学習の書式: `変数.fit(データ)`
* クラスタリングした結果（クラスタ番号）は，オブジェクトの`labels_`属性から確認できる
  * 書式: `変数.labels_`
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3～7行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 8行目:  `sklearn` (scikit-learn) の`cluster`モジュール内にある`KMeans`クラスを読み込む 
* 9行目: `KMeans(n_clusters=3, random_state=random_seed)`で，モデルの学習を行うためのオブジェクトを`KMeans`クラスから生成し，変数`kmeans_model`に代入
  * `n_clusters=3`で，クラスタ数を 3 に設定
  * `random_state=random_seed`で，乱数を固定
* 10行目: `fit`メソッドでモデルの学習（データ`df_cluster`のクラスタリング）を実行
* 11行目: クラスタリングした結果の列データ（`kmeans_model.labels_`）を`df_cluster`に追加
  * 列名を「cluster」とした
* 12行目: `display`関数を使ってデータ（`df_cluster`の内容）を表示

In [ ]:
# モデルの学習





df_cluster = pd.DataFrame(df_s, columns=df.columns)
from sklearn.cluster import KMeans
kmeans_model = KMeans(n_clusters=3, random_state=random_seed)
kmeans_model.fit(df_cluster)
df_cluster['cluster'] = kmeans_model.labels_
display(df_cluster)

## モデルの分析
* クラスタリングができたので，各クラスタに属するデータの特徴などを分析する

### 各クラスタのデータ数
* `value_counts`メソッドを使って，各クラスタのデータ数を確認する
* `value_counts`メソッドについては，第3回「[サンプルノートブック03](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook03.ipynb)」や共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3～7行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 8～11行目: データのクラスタリングを行い，その結果を`df_cluster`に代入
* 12行目: クラスタリングの結果が格納されている cluster列の値の頻度を抽出し，その結果を`display`関数で表示

In [ ]:
# モデルの分析





df_cluster = pd.DataFrame(df_s, columns=df.columns)
from sklearn.cluster import KMeans
kmeans_model = KMeans(n_clusters=3, random_state=random_seed)
kmeans_model.fit(df_cluster)
df_cluster['cluster'] = kmeans_model.labels_
display(pd.DataFrame(df_cluster['cluster'].value_counts()))

### 各クラスタにおける特徴量の平均値
* `groupby`メソッドと平均値を計算する`mean`メソッドを使って，各クラスタにおける特徴量の平均値を計算する
* `groupby`メソッドについては，第5回「[サンプルノートブック05](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook05.ipynb)」や共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3～7行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 8～11行目: データのクラスタリングを行い，その結果を`df_cluster`に代入
* 12行目: 各クラスタにおける各特徴量の平均値を変数`cluster_mean`に代入
  * 平均値は`mean`メソッドを用いて計算 ⇒ `df_cluster.groupby('cluster').mean()`
* 13行目: `DataFrame`関数で`cluster_mean`をDataFrameに変換し，その結果を`display`関数で表示
  * `groupby`メソッドによる集計結果はSeriesとなる

In [ ]:
# 各クラスタにおける特徴量の平均値





df_cluster = pd.DataFrame(df_s, columns=df.columns)
from sklearn.cluster import KMeans
kmeans_model = KMeans(n_clusters=3, random_state=random_seed)
kmeans_model.fit(df_cluster)
df_cluster['cluster'] = kmeans_model.labels_
cluster_mean = df_cluster.groupby('cluster').mean()
display(pd.DataFrame(cluster_mean))

### 各特徴量の平均値の可視化
* 各クラスタの平均値を棒グラフで可視化する
* 棒グラフは，DataFrameの`plot`メソッドを使って作成する
* `plot`メソッドについては，共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3～7行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 8～11行目: データのクラスタリングを行い，その結果を`df_cluster`に代入
* 12行目: 各クラスタにおける各特徴量の平均値を変数`cluster_mean`に代入
  * 平均値は`mean`メソッドを用いて計算 ⇒ `df_cluster.groupby('cluster').mean()`
* 13行目: `plot`メソッドを使って，各クラスタの平均値`cluster_mean`の棒グラフを作成
  * `kind='bar'`で，グラフの種類を棒グラフに設定
  * `title='各クラスタの平均値'`で，グラフのタイトルを「各クラスタの平均値」に設定
* 14行目: `show`関数で，それまでに設定したグラフを実行画面に表示

In [ ]:
# 各特徴量の平均値の可視化
df = pd.read_csv('wholesale.csv')
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc.fit(df)
df_s = sc.transform(df)
df_cluster = pd.DataFrame(df_s, columns=df.columns)
from sklearn.cluster import KMeans
kmeans_model = KMeans(n_clusters=3, random_state=random_seed)
kmeans_model.fit(df_cluster)
df_cluster['cluster'] = kmeans_model.labels_
cluster_mean = df_cluster.groupby('cluster').mean()
cluster_mean.plot(kind='bar', title='各クラスタの平均値')
plt.show()

# エルボー法によるクラスタ数の決定
* k-means法では分析者が事前に何個のクラスタに分けるかを決定する必要がある
* クラスタ数の決め方の一つにエルボー法がある
* エルボー法の手順
  * クラスタ数を事前に見積もる方法の1つ
  * クラスタ数kを変化させたときの，クラスタ内誤差平方和（SSE: Sum of Squared Errors）の減少度合をみて適切なkを決める
  * SSE: 各クラスタの重心点（平均）とそのクラスタに属するデータの差（距離）の2乗の総和 ⇒ kが増加するとSSEは減少する
  * SSEの減少度合が大きく低下している k をクラスタ数とする
* 以下のコードで，クラスタ数2～30のSSEを計算し，その結果をグラフ化している
* コードの内容は理解しなくてよいので，実行結果だけ確認しておく

In [ ]:
# エルボー法によるクラスタ数の決定
df = pd.read_csv('wholesale.csv')
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc.fit(df)
df_s = sc.transform(df)
df_cluster = pd.DataFrame(df_s, columns=df.columns)
from sklearn.cluster import KMeans
sse_list = []
for n in range(2, 31):
    kmeans_model = KMeans(n_clusters=n, random_state=random_seed)
    kmeans_model.fit(df_cluster)
    sse_list.append(kmeans_model.inertia_)
plt.plot(range(2,31), sse_list, marker='o')
plt.xlabel('クラスタ数')
plt.ylabel('SSE（クラスタ内誤差平方和）')
plt.show()

# クラスタリングの分析例：アヤメデータ

## 用いるデータと分析の目的
* 例として，csvファイル「iris.csv」を用いる
  * 出典: 須藤秋良, 株式会社フレアリンク: スッキリわかるPythonによる機械学習入門, インプレス, 2020
* アヤメデータの2つの特徴量「花弁長さ」と「花弁幅」からなるデータのクラスタリングを行う

## 実習：データの前処理
* 以下のコードで，アヤメデータの前処理を行う
* 前処理の内容
  * 欠損値を各特徴量の平均値でそれぞれ置き換える（第6回「[サンプルノートブック06](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook06.ipynb)」と実習と同様）
  * 「花弁長さ」と「花弁幅」のデータを抽出
  * データの標準化を行う

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `fillna`メソッドと`mean`メソッドを使って，欠損値を各説明変数（がく片長さ, がく片幅, 花弁長さ, 花弁幅）の平均値にそれぞれ置き換えて変数`df`に代入
* 4行目: データの列名を要素とするリスト`['花弁長さ', '花弁幅']`を変数`cols`に代入
* 5行目: DataFrame `df` から,`df[cols]`で，「花弁長さ」と「花弁幅」の列だけ取り出し変数`df_cluster`に代入
* 6行目: `preprocessing`モジュールの`StandardScaler`クラスの読み込み
* 7行目: `StandardScaler`クラスのオブジェクトを生成し，変数`sc`に代入
* 8行目: `sc`に対し，`fit`メソッドを適用する（引数は`df_cluster`）
* 9行目: `sc`に対する`transform`メソッドで`df_cluster`を標準化し，その結果を変数`df_s`に代入
* 10行目: `DataFrame`関数で，`df_s`をDataFrameに変換し，その結果を変数`df_cluster`に代入
  * `columns=cols`で，変換したDataFrameの列名を「花弁長さ」と「花弁幅」（`cols`）に指定
* 11行目: `display`関数で，データ`df_cluster`の内容を表示

**［実行結果］**

  <img src="./fig/exercise_preprocessing_iris.jpg" width="170">

In [ ]:
# アヤメデータの前処理








df_cluster = pd.DataFrame(df_s, columns=cols)
display(df_cluster)

## 実習：モデルの学習（k-means法によるクラスタリング）
**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3行目: `fillna`メソッドと`mean`メソッドを使って，欠損値を各説明変数（がく片長さ, がく片幅, 花弁長さ, 花弁幅）の平均値にそれぞれ置き換えて変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 4～5行目: `df` からデータの列`['花弁長さ', '花弁幅']`だけ取り出し変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～10行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 11行目: `sklearn` (scikit-learn) の`cluster`モジュール内にある`KMeans`クラスを読み込む 
* 12行目: モデルの学習を行うためのオブジェクトを`KMeans`クラスから生成し，変数`kmeans_model`に代入
  * クラスタ数を 3 に設定
  * `random_state=random_seed`で，乱数を固定
* 13行目: `fit`メソッドでモデルの学習（データ`df_cluster`のクラスタリング）を実行
* 14行目: クラスタリングした結果の列データ（`kmeans_model.labels_`）を`df_cluster`に追加
  * 列名を「cluster」とした
* 15行目: `display`関数を使ってデータ（`df_cluster`の内容）を表示

**［実行結果］**

  <img src="./fig/exercise_clustering_iris.jpg" width="240">

In [ ]:
# モデルの学習













display(df_cluster)

## 各クラスタのデータ数
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3行目: `fillna`メソッドと`mean`メソッドを使って，欠損値を各説明変数（がく片長さ, がく片幅, 花弁長さ, 花弁幅）の平均値にそれぞれ置き換えて変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 4～5行目: `df` からデータの列`['花弁長さ', '花弁幅']`だけ取り出し変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～10行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 11～14行目: データのクラスタリングを行い，その結果を`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 15行目: `value_counts`メソッドで，クラスタリングの結果が格納されている cluster列の値の頻度を抽出し，その結果を`display`関数で表示

In [ ]:
# 各クラスタのデータ数













display(pd.DataFrame(df_cluster['cluster'].value_counts()))

## クラスタと種類のクロス集計
* クラスタリングによって分割された各クラスタのデータとアヤメの種類がどれくらい一致しているかを確認する
* 具体的には，クラスタと種類に対して，組み合わせ頻度を数えるクロス集計を行う
* 組み合わせ頻度を数えるクロス集計は，pandas（`pd`）の`crosstab`関数を用いる
* `crosstab`関数については，共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
* 以下のコードで，組み合わせ頻度を数えるクロス集計を行った結果，setosa はクラスタ1，versicolor はクラスタ0，virginica はクラスタ2にそれぞれ分かれている傾向が見られる
* 特に setosa は完全に分離されていることがわかる
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3行目: `fillna`メソッドと`mean`メソッドを使って，欠損値を各説明変数（がく片長さ, がく片幅, 花弁長さ, 花弁幅）の平均値にそれぞれ置き換えて変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 4～5行目: `df` からデータの列`['花弁長さ', '花弁幅']`だけ取り出し変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～10行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 11～14行目: データのクラスタリングを行い，その結果を`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 15行目: `df_cluster`に種類の列 `df['種類']` を追加する
* 16行目: `crosstab`メソッドを使って，cluster列（`df_cluster['cluster']`）と種類列（`df_cluster['種類']`）の組み合わせ頻度を数えるクロス集計を行い，その結果（DataFrame）を変数`ct`に代入
* 17行目: `display`関数で，クロス集計の結果 `ct` を表示

In [ ]:
# クラスタと種類のクロス集計













df_cluster['種類'] = df['種類']
ct = pd.crosstab(df_cluster['cluster'], df_cluster['種類'])
display(ct)

## 散布図の作成
* クロス集計の結果を視覚的に確認するため，花弁長さと花弁幅に対する散布図を2つ作成する
>* 一方はクラスタごとにマーカーを色分けした散布図
>* もう一方は種類ごとにマーカーを色分けした散布図
* 散布図は，seaborn（`sns`）の`lmplot`関数を用いて作成する
* 書式: `sns.lmplot(data=[DataFrame], x=[x軸の列名], y=[y軸の列名], hue=[色分けしたい列名], fit_reg=False)`
  * `data`には，散布図を作成する対象のDataFrameを指定
  * `x`には，x軸に設定する列の列名
  * `y`には，y軸に設定する列の列名
  * `hue`には，色分けしたい列の列名
  * `fit_reg=False`で，回帰直線を非表示
  * `lmplot`関数は，散布図上に回帰直線を描くことができるが，ここでは必要ないので`fit_reg=False`とする
  * `fit_reg=True`（デフォルト）にすると，`hue`で指定した列の値ごとの回帰直線が表示される
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 3行目: `fillna`メソッドと`mean`メソッドを使って，欠損値を各説明変数（がく片長さ, がく片幅, 花弁長さ, 花弁幅）の平均値にそれぞれ置き換えて変数`df`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 4～5行目: `df` からデータの列`['花弁長さ', '花弁幅']`だけ取り出し変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～10行目: データの標準化を行い，標準化したデータを変数`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 11～14行目: データのクラスタリングを行い，その結果を`df_cluster`に代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 15行目: `df_cluster`に種類の列 `df['種類']` を追加する
* 16行目: `lmplot`関数を使って，クラスタごとに色分けした散布図を作成
* 17行目: `show`関数で，それまでに設定した図（クラスタごとに色分けした散布図）を実行画面に表示
* 18行目: `lmplot`関数を使って，種類ごとに色分けした散布図を作成
* 19行目: `show`関数で，それまでに設定した図（種類ごとに色分けした散布図）を実行画面に表示

In [ ]:
# 散布図の作成













df_cluster['種類'] = df['種類']
sns.lmplot(data=df_cluster, x='花弁長さ', y='花弁幅', hue='cluster', fit_reg=False)
plt.show()
sns.lmplot(data=df_cluster, x='花弁長さ', y='花弁幅', hue='種類', fit_reg=False)
plt.show()